> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# 04 | Dynamic Panel and Temporal Regression — Hotel Labour Productivity in Portugal

**Luís Silvano's doctoral research project · Preparation of a paper for *Knowledge-Based Systems***  
**Data period:** 2018–2024 | **Unit of analysis:** hotel × year | **Task:** continuous regression

### Scientific question

> **Does temporal dependency improve hotel labor productivity forecasting beyond structural characteristics and spatial context, particularly under pandemic-induced economic disruptions?**

This notebook **extends**, rather than replaces, Notebook 03. The focus is on determining whether past productivity helps predict productivity in the following year and whether spatial information remains useful after including this temporal memory.

**Research hypotheses**

- **H1 — Temporal dependence:** models using the previous year's productivity outperform models with the same structural and spatial characteristics but without that history.
- **H2 — Pandemic shock:** the usefulness of historical information changes during the pandemic and recovery period; whether 2020 should be included in training requires investigation.
- **H3 — Spatial context:** information about hotels' surroundings adds predictive value even after including historical productivity.

**Important:** better forecasts and statistical associations **are not causal evidence**. Results for 2024 were consulted in earlier notebooks; here they constitute a **retrospective audit**, not a genuinely untouched test.


## Quick guide for Luís

1. Place this `.ipynb` and `2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx` **in the same folder**. No changes to the Excel file are required.
2. In the **same Jupyter kernel**, install the core packages if necessary:
   ```python
   %pip install pandas numpy matplotlib scikit-learn openpyxl lxml scipy statsmodels xgboost lightgbm
   ```
3. To include the modern TabM and RealMLP models:
   ```python
   %pip install 'pytabkit[models]'
   ```
   These models are **optional** and may take a long time, particularly without a GPU. If the library fails, the notebook records the reason and continues with the remaining models.
4. Select `PERFIL = 'completo'` or `'verificacao'` in the next cell. The former **requests** modern models; the latter is recommended for quickly checking the environment.
5. Run **Kernel → Restart & Run All** and inspect the `resultados_KBS_04_painel_dinamico` folder.

**Glossary:** *lag-1* = previous-year value; *ARX* = model combining the target's history with external factors; *walk-forward* = successive evaluation in future years; *panel* = multiple hotels observed over several years; *ablation* = removing a group of variables to measure its contribution.

> **Note on ARIMAX/SARIMAX:** we have at most seven **annual observations per hotel**. Fitting a hotel-specific seasonal SARIMAX with such short histories would not be scientifically sound. Instead, we use pooled ARX and panel models that are comparable and suitable for these data; individual ARIMAX/SARIMAX analysis is documented as a limitation rather than given an artificial ranking result.


## 1. Reproducible configuration — change only this cell


In [ ]:
from pathlib import Path
import time, sys, json, traceback, importlib.util, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import sparse

SEMENTE = 42
np.random.seed(SEMENTE)
FICHEIRO_DADOS = Path('2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx')
FOLHA = 'New Dataset'
PASTA_SAIDA = Path('resultados_KBS_04_painel_dinamico')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PERFIL = 'completo'          # 'verificacao' (quick check) or 'completo' (requests TabM and RealMLP)
ANOS_CV = [2021, 2022, 2023]
ANO_AUDITORIA = 2024        # already examined in previous experiments!
EXECUTAR_AUDITORIA_2024 = True
EXECUTAR_SENSIBILIDADE = True
EXECUTAR_PAINEL_FE_DESCRITIVO = True  # requires linearmodels; econometric report only, not part of the ranking
CENARIO_PRINCIPAL = 'todos_p995'    # defined BEFORE the dynamic benchmark
CENARIOS_SENSIBILIDADE = ['todos_original', 'todos_p995', 'sem2020_original', 'sem2020_p995']
MAX_THREADS = 2
N_BOOTSTRAPS = 350

ID_HOTEL = 'ht_id'
COL_ANO = 'f_year'
COL_ALVO = 'f_labprodct'
COL_CADEIA = 'ht_hotel_chain_d'
ESTRUTURA = ['ht_capacity', 'ht_log_number_of_accommodation_units', 'ht_stars_classification']
ESPACO = ['ht_location_beach1', 'ht_location_castle1', 'ht_location_church1', 'ht_Restaurants&CoffesL800m']
ATRIBUTOS_HOTEL = ESTRUTURA + ESPACO
COLUNAS_AUDITORIA = ['f_number_of_employees', 'f_operating_revenue_turnover2']
# As a safeguard, revenue and employee counts from the prediction year are NOT included.
# Even t-1 financial data may be published with a delay: include only after verifying availability.

MODELOS_BASE = ['media_treino', 'persistencia', 'ols_arx',
                'ridge_static', 'ridge_arx', 'ridge_arx_lagclip', 'ridge_panel_arx', 'ridge_arx_sem_espaco',
                'random_forest_static', 'random_forest_arx', 'random_forest_arx_lagclip',
                'xgboost_static', 'xgboost_arx', 'xgboost_arx_lagclip', 'xgboost_arx_sem_espaco', 'lightgbm_arx']
MODELOS_AVANCADOS = ['tabm_static', 'tabm_arx', 'realmlp_arx']
MODELOS = MODELOS_BASE + (MODELOS_AVANCADOS if PERFIL == 'completo' else [])
assert PERFIL in {'verificacao', 'completo'}
assert CENARIO_PRINCIPAL in CENARIOS_SENSIBILIDADE
assert max(ANOS_CV) < ANO_AUDITORIA
print('Excel path:', FICHEIRO_DADOS.resolve())
print('Profile:',PERFIL, '| requested models:', len(MODELOS))
print('Primary scenario:', CENARIO_PRINCIPAL)

## 2. Safely reading the original Excel file

We read only the necessary columns, verifying the **actual file and worksheet** before proceeding. This step uses the fast reader validated in the previous project version and **does not modify the Excel file**. Financial variables appear in the audit only and are not inadvertently included in predictions.


In [ ]:
# This cell explicitly verifies WHICH file was read and WHICH worksheet was selected.
# The fast reader loads only necessary columns and does not modify the original Excel file.
import re
import zipfile
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

FOLHA_DADOS = FOLHA
COLUNAS_NECESSARIAS = list(dict.fromkeys(
    [ID_HOTEL, COL_ANO, COL_ALVO, COL_CADEIA] + ATRIBUTOS_HOTEL + COLUNAS_AUDITORIA
))

if not FICHEIRO_DADOS.exists():
    raise FileNotFoundError(
        f'Excel file not found: {FICHEIRO_DADOS.resolve()}\n'
        'Place the Excel file next to this notebook or edit FICHEIRO_DADOS in Section 1.\n'
        f'Current Jupyter directory: {Path.cwd()}'
    )

print('Current Jupyter directory:', Path.cwd())
print('File loaded (absolute path):', FICHEIRO_DADOS.resolve())
print('File size (MB):', round(FICHEIRO_DADOS.stat().st_size / 1024**2, 2))

if FICHEIRO_DADOS.suffix.lower() in ('.xlsx', '.xlsm'):
    pasta_excel = load_workbook(FICHEIRO_DADOS, read_only=True, data_only=True)
    folhas = pasta_excel.sheetnames
    print('Worksheets found:', folhas)
    if FOLHA_DADOS not in folhas:
        pasta_excel.close()
        raise ValueError(
            f'Worksheet {FOLHA_DADOS!r} was not found in this Excel file. '
            f'Available worksheets: {folhas}. Please verify that you selected the correct file.'
        )
    folha = pasta_excel[FOLHA_DADOS]
    cabecalho_excel = [str(v).strip() if v is not None else ''
                       for v in next(folha.iter_rows(min_row=1, max_row=1, values_only=True))]
    pasta_excel.close()
    print('Columns in the original Excel file:', len(cabecalho_excel))

    colunas_em_falta = [c for c in COLUNAS_NECESSARIAS if c not in cabecalho_excel]
    if colunas_em_falta:
        print('EXAMPLES OF AVAILABLE COLUMNS:', cabecalho_excel[:25])
        raise ValueError(
            'The following columns were not found in the OPENED file:\n  - '
            + '\n  - '.join(colunas_em_falta)
            + '\nCheck the absolute path above and the Excel file version. '
              'These columns are present in the original Excel file provided by the research team. '
              'These variables will not be silently replaced with others.'
        )

    def read_selected_excel_columns(caminho, nomes, cabecalho):
        'Reads only the requested columns, avoiding the processing of millions of cells in pandas.\n\n        Falls back to pandas.read_excel when lxml is unavailable.\n        Does not modify formulas or values in the original Excel file.\n        '
        try:
            from lxml import etree as ET
        except ImportError:
            print('lxml is not installed; standard reading may take several minutes.')
            return pd.read_excel(
                caminho, sheet_name=FOLHA_DADOS, usecols=lambda c: c.strip() in nomes,
                engine='openpyxl'
            )

        # This project version has one worksheet. For Excel files with multiple
        # worksheets, the standard reader unambiguously resolves the worksheet name.
        if len(folhas) != 1:
            print('Excel file with multiple worksheets: reading with pandas.')
            return pd.read_excel(
                caminho, sheet_name=FOLHA_DADOS, usecols=lambda c: c.strip() in nomes,
                engine='openpyxl'
            )

        ns = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
        # Map an Excel column letter (e.g., 'A', 'HQ') to the variable name.
        letra_para_nome = {
            get_column_letter(i+1): nome
            for i, nome in enumerate(cabecalho)
            if nome in nomes
        }
        linhas = []
        with zipfile.ZipFile(caminho) as arq:
            texto_compartilhado = []
            if 'xl/sharedStrings.xml' in arq.namelist():
                raiz = ET.parse(arq.open('xl/sharedStrings.xml')).getroot()
                texto_compartilhado = [
                    ''.join(si.itertext()) for si in raiz.findall(ns + 'si')
                ]
            with arq.open('xl/worksheets/sheet1.xml') as fluxo:
                parser = ET.iterparse(fluxo, events=('end',), tag=ns+'row', huge_tree=True)
                for _, linha in parser:
                    if linha.get('r') != '1':
                        registo = {}
                        for celula in linha.iterchildren(tag=ns+'c'):
                            # Cell A12 -> column A; the row number is not needed.
                            letra = celula.get('r', '').rstrip('0123456789')
                            nome = letra_para_nome.get(letra)
                            if nome is None:
                                continue
                            valor = celula.findtext(ns+'v')
                            if valor is None:
                                continue
                            tipo = celula.get('t')
                            if tipo == 's':
                                valor = texto_compartilhado[int(valor)]
                            elif tipo == 'b':
                                valor = (valor == '1')
                            elif tipo not in ('str', 'inlineStr'):
                                try:
                                    valor = float(valor)
                                except (ValueError, TypeError):
                                    pass
                            registo[nome] = valor
                        linhas.append(registo)
                    # Avoid storing all three million Excel cells in memory.
                    linha.clear()
                    while linha.getprevious() is not None:
                        del linha.getparent()[0]
        return pd.DataFrame(linhas, columns=nomes)

    dados_brutos = read_selected_excel_columns(
        FICHEIRO_DADOS, COLUNAS_NECESSARIAS, cabecalho_excel
    )
elif FICHEIRO_DADOS.suffix.lower() == '.csv':
    cabecalho_excel = pd.read_csv(FICHEIRO_DADOS, nrows=0).columns.tolist()
    dados_brutos = pd.read_csv(FICHEIRO_DADOS, usecols=COLUNAS_NECESSARIAS)
else:
    raise ValueError('Unrecognised format. Use .xlsx, .xlsm, or .csv.')

dados_brutos.columns = [str(c).strip() for c in dados_brutos.columns]
print('Worksheet used:', FOLHA_DADOS if FICHEIRO_DADOS.suffix.lower() != '.csv' else 'CSV')
print('Rows read:', len(dados_brutos))
print('Columns loaded for the study:', list(dados_brutos.columns))
print('NOTE: only relevant variables are loaded; the original Excel file contains 277 columns.')
display(dados_brutos.head(3))

## 3. Data integrity audit and proper temporal panel construction

To predict year **t**, we use only information from **year t−1** for the same hotel. Lags are constructed by an explicit `hotel + year` join; a hotel observed in 2021 and then in 2023 **will not** receive its 2021 productivity as though it came from 2022.

**The comparison cohort is identical across all models:** only observations with valid productivity in both year *t* and year *t−1* are included. This prevents the dynamic model from being unfairly penalised or favoured by evaluation on a different group of hotels. One unavoidable drawback is that hotels without a historical observation are excluded from this particular benchmark; we will report that selection.

Structural and spatial characteristics are also shifted to year *t−1*. **Data provenance matters:** if distances or restaurant counts were retrospectively calculated from a more recent inventory, that must be acknowledged in the manuscript, even if the same feature values have been repeated across the historical panel.


In [ ]:
for c in [COL_ANO, COL_ALVO, COL_CADEIA] + ATRIBUTOS_HOTEL + COLUNAS_AUDITORIA:
    dados_brutos[c] = pd.to_numeric(dados_brutos[c], errors='coerce')
dados_brutos = dados_brutos.replace([np.inf,-np.inf], np.nan)
indep = dados_brutos.loc[(dados_brutos[COL_CADEIA] == 0) &
                         (dados_brutos[COL_ANO].between(2018, 2024))].copy()
indep = indep.dropna(subset=[ID_HOTEL,COL_ANO]).copy()
indep[COL_ANO] = indep[COL_ANO].astype(int)
indep[ID_HOTEL] = indep[ID_HOTEL].astype('Int64').astype(str)
if indep.duplicated([ID_HOTEL,COL_ANO]).any():
    raise ValueError('Duplicate hotel–year pairs! Review the Excel file before running the models.')
if (indep[COL_ALVO].dropna() < 0).any():
    raise ValueError('Negative productivity values found; review the definition of this indicator.')

validos = indep.dropna(subset=[COL_ALVO]).copy()
cols_anteriores = [COL_ALVO] + ATRIBUTOS_HOTEL
anterior = indep[[ID_HOTEL,COL_ANO] + cols_anteriores].copy()
anterior[COL_ANO] += 1
anterior = anterior.rename(columns={COL_ALVO:'produtividade_lag1',
                 **{col:'ano_anterior__'+col for col in ATRIBUTOS_HOTEL}})

# Only the current-year target appears on the left-hand side of the prediction.
pares = validos[[ID_HOTEL, COL_ANO, COL_ALVO]].merge(
    anterior,on=[ID_HOTEL,COL_ANO],how='left',validate='one_to_one'
)
pares = pares.dropna(subset=['produtividade_lag1']).copy()
pares = pares.sort_values([COL_ANO, ID_HOTEL]).reset_index(drop=True)
pares['ano_previsao_centrado'] = pares[COL_ANO] - 2018
# Do not treat the pandemic as information known in advance: regime indicators
# are used for year-by-year DESCRIPTIVE analysis only, not as predictors.
pares['ano_anterior_foi_2020'] = (pares[COL_ANO] - 1 == 2020).astype(int)
ATRIBUTOS_EST = ['ano_anterior__'+col for col in ESTRUTURA] + ['ano_previsao_centrado']
ATRIBUTOS_ESP = ['ano_anterior__'+col for col in ESPACO]
ATRIBUTOS_STATIC = ATRIBUTOS_EST + ATRIBUTOS_ESP
ATRIBUTOS_ARX = ATRIBUTOS_STATIC + ['produtividade_lag1']
ATRIBUTOS_ARX_SEM_ESPACO = ATRIBUTOS_EST + ['produtividade_lag1']

assert pares['produtividade_lag1'].notna().all()
assert pares[COL_ANO].min() >= 2019
assert not pares.duplicated([ID_HOTEL,COL_ANO]).any()
print('Observations in the Excel file:',len(dados_brutos))
print('Independent-hotel observations:',len(indep),'| unique hotels:',indep[ID_HOTEL].nunique())
print('Observations with a known target:',len(validos))
print('Consecutive-year pairs eligible for dynamic evaluation:',len(pares))
print('Pairs by year:')
contagens = pares.groupby(COL_ANO).agg(observacoes=(ID_HOTEL,'size'),hoteis=(ID_HOTEL,'nunique'))
display(contagens)
assert set(ANOS_CV+[ANO_AUDITORIA]).issubset(set(pares[COL_ANO]))

## 4. Describe the pandemic **before** modelling

This analysis addresses the question: did productivity change among **the same hotels** during the pandemic? Describing a decline does not by itself establish causality; other economic changes may have occurred simultaneously.


In [ ]:
por_ano=validos.groupby(COL_ANO).agg(
    n=(COL_ALVO,'size'),media=(COL_ALVO,'mean'),mediana=(COL_ALVO,'median'),
    q25=(COL_ALVO,lambda x:x.quantile(.25)),q75=(COL_ALVO,lambda x:x.quantile(.75)),
    q99=(COL_ALVO,lambda x:x.quantile(.99))
).reset_index()
fig,ax=plt.subplots(figsize=(9,4.6))
ax.plot(por_ano[COL_ANO],por_ano['mediana'],marker='o',label='Median')
ax.plot(por_ano[COL_ANO],por_ano['media'],marker='s',label='Mean')
ax.fill_between(por_ano[COL_ANO],por_ano['q25'],por_ano['q75'],alpha=.16,label='Intervalo interquartil')
ax.axvspan(2019.65,2020.35,alpha=.12,label='2020 (pandemic year)')
ax.set(title='Annual labour productivity — independent hotels',xlabel='Year',ylabel='Productivity (original units)')
ax.legend();fig.tight_layout()
fig.savefig(PASTA_SAIDA/'figura_01_evolucao_anual.png',dpi=190)
plt.show()

y19=validos.loc[validos[COL_ANO]==2019,[ID_HOTEL,COL_ALVO]].rename(columns={COL_ALVO:'y2019'})
y20=validos.loc[validos[COL_ANO]==2020,[ID_HOTEL,COL_ALVO]].rename(columns={COL_ALVO:'y2020'})
p19_20=y19.merge(y20,on=ID_HOTEL,validate='one_to_one')
print('Hotels observed in both 2019 and 2020:',len(p19_20))
print('Percentage with a decline from 2019 to 2020:',round((p19_20.y2020<p19_20.y2019).mean()*100,1),'%')
print('Do not interpret these descriptive findings as causal evidence of the pandemic.')

## 5. Models and their interpretation

| Family | Models | Question to investigate |
|---|---|---|
| Baselines | Historical mean; persistence | Are predictions better than simply repeating last year's productivity? |
| Dynamic statistics | **OLS-ARX** and **Ridge-ARX** | What is the incremental value of historical productivity after controlling for covariates? |
| Predictive panel | **Ridge-ARX + penalised hotel-specific intercepts** | Is there stable heterogeneity among hotels? |
| Machine learning | Random Forest, XGBoost, LightGBM | Does nonlinearity improve temporal forecasts? |
| Tabular deep learning | TabM and RealMLP (optional) | Do modern neural models benefit from the same dynamic information? |

**Pooled ARX:** `productivity_t = intercept + rho * productivity_(t-1) + beta * characteristics_(t-1) + error_t`.

**ARX-lagclip (robustness):** in addition to limiting extreme training-target values, we clip **lag-1** at the 99.5th percentile computed **exclusively from the training data**. This analysis tests whether exceptionally high productivity in 2022 distorts predictions made for 2023. The actual evaluated values remain untouched.

**Panel with penalised intercepts:** we add hotel indicators to a Ridge regression so that the model can capture different persistent productivity levels. **This is not equivalent to a causal dynamic-panel estimator.** An exploratory fixed-effects model using `linearmodels` (optional) is fitted at the end: with a small T and a lagged dependent variable, dynamic-panel bias (Nickell bias) arises. Fixed effects also absorb spatial attributes that do not vary within hotels.

**ARIMAX versus ARX:** with only seven years per hotel, unit-specific ARIMA/ARIMAX is poorly identified. For this paper, ARX models that share information across hotels are more suitable. Seasonal SARIMAX is **not** appropriate for such short annual series.


## 6. Walk-forward evaluation — as if forecasting at the end of t−1

- **2021:** fit on 2019–2020 pairs, predict 2021;
- **2022:** fit on 2019–2021 pairs, predict 2022;
- **2023:** fit on 2019–2022 pairs, predict 2023;
- **2024:** retrospective audit only, fit on 2019–2023.

The same observations from the forecast year are used by **all** models, including static ones. Outlier treatment (p99.5) changes **only the training target**. The test retains the true productivity, including all extreme values.

The main evaluation is *one-step-ahead*: predicting 2024 assumes that 2023 productivity is available. If financial indicators are released with a delay, the operational forecast date must be adjusted accordingly.


In [ ]:
def split_by_year(ano):
    tr=pares.loc[pares[COL_ANO]<ano].copy()
    va=pares.loc[pares[COL_ANO]==ano].copy()
    assert len(tr)>0 and len(va)>0
    assert tr[COL_ANO].max() < ano
    assert va[COL_ANO].nunique()==1
    # Features used in each validation row always come from the preceding year.
    assert (va[COL_ANO].to_numpy()-1>=2018).all()
    return tr,va

for ano in ANOS_CV+[ANO_AUDITORIA]:
    tr,va=split_by_year(ano)
    print(f'{ano}: training = {len(tr)} rows / validation = {len(va)} rows / last training year = {tr[COL_ANO].max()}')

## 7. Preprocessing and training functions

Missing feature values are imputed using the **median computed only from training data**. For Ridge and OLS, numerical features are also standardised using training data only. Final predictions are lower-bounded by zero, because productivity is nonnegative, using the same rule across all models.

If an optional model fails, we save the exception in a report: **a requested model is not automatically an executed model**.


In [ ]:
FALHAS=[]
def record_failure(etapa,modelo,ano,cenario,exc):
    registro={'etapa':etapa,'modelo':modelo,'ano':ano,'cenario':cenario,
              'erro':str(exc)[:450],'tipo':type(exc).__name__,
              'traceback':traceback.format_exc(limit=4)[:1100]}
    FALHAS.append(registro)
    print('NOT EXECUTED:',modelo,'|',ano,'|',str(exc)[:135])

def calculate_metrics(y,p):
    y=np.asarray(y,dtype=float).ravel();p=np.asarray(p,dtype=float).ravel()
    if len(y)!=len(p) or len(y)==0 or not np.isfinite(p).all():
        raise ValueError('Predictions are empty, have an incorrect shape, or contain non-finite values.')
    return {'MAE':float(mean_absolute_error(y,p)),
            'RMSE':float(np.sqrt(mean_squared_error(y,p))),
            'R2':float(r2_score(y,p))}

def prepare_sample(tr,cenario):
    assert cenario in CENARIOS_SENSIBILIDADE
    tr=tr.copy()
    if cenario.startswith('sem2020'):
        tr=tr.loc[tr[COL_ANO]!=2020].copy()
    if len(tr)<60:
        raise ValueError('Insufficient training sample for this scenario.')
    yy=tr[COL_ALVO].astype(float).to_numpy(copy=True)
    limite=None
    if cenario.endswith('p995'):
        limite=float(np.quantile(yy,.995))
        yy=np.minimum(yy,limite)
    return tr,yy,limite

def get_feature_matrices(tr,va,features,standardize=False):
    if not features: raise ValueError('Empty feature list.')
    vazias=[x for x in features if tr[x].notna().sum()==0]
    if vazias: raise ValueError('Features entirely missing from training data: '+str(vazias))
    imp=SimpleImputer(strategy='median')
    Xtr=imp.fit_transform(tr[features].astype(float))
    Xva=imp.transform(va[features].astype(float))
    if standardize:
        s=StandardScaler();Xtr=s.fit_transform(Xtr);Xva=s.transform(Xva)
    assert np.isfinite(Xtr).all() and np.isfinite(Xva).all()
    return Xtr.astype('float32'),Xva.astype('float32')

def get_model_details(nome):
    if nome=='media_treino': return [],'dummy'
    if nome=='persistencia': return ['produtividade_lag1'],'persistencia'
    if nome.endswith('_sem_espaco'): return ATRIBUTOS_ARX_SEM_ESPACO,'arx_sem_espaco'
    if nome.endswith('_static'): return ATRIBUTOS_STATIC,'static'
    return ATRIBUTOS_ARX,'arx'

def fit_and_predict(nome,treino,validacao,cenario=CENARIO_PRINCIPAL):
    tr,y_train,limite=prepare_sample(treino,cenario)
    features,grupo=get_model_details(nome)
    inicio=time.perf_counter()
    limite_lag=None
    tr_features=tr
    va_features=validacao
    if nome.endswith('_lagclip'):
        limite_lag=float(np.quantile(tr['produtividade_lag1'],.995))
        tr_features=tr.copy();va_features=validacao.copy()
        tr_features['produtividade_lag1']=tr_features['produtividade_lag1'].clip(upper=limite_lag)
        va_features['produtividade_lag1']=va_features['produtividade_lag1'].clip(upper=limite_lag)
    if nome=='media_treino':
        pred=np.full(len(validacao),float(np.mean(y_train)))
    elif nome=='persistencia':
        pred=validacao['produtividade_lag1'].to_numpy(dtype=float)
    else:
        usar_escala=nome.startswith(('ridge_','ols_'))
        Xtr,Xva=get_feature_matrices(tr_features,va_features,features,standardize=usar_escala)
        if nome=='ols_arx':
            import statsmodels.api as sm
            Xtr_const=sm.add_constant(Xtr,has_constant='add')
            Xva_const=sm.add_constant(Xva,has_constant='add')
            # Cluster standard errors by hotel because the panel has repeated measures.
            mod=sm.OLS(y_train,Xtr_const).fit(cov_type='cluster',
                     cov_kwds={'groups':tr[ID_HOTEL].astype(str).to_numpy()})
            pred=mod.predict(Xva_const)
        elif nome=='ridge_panel_arx':
            # Hotel-specific intercepts with shrinkage (penalised effects).
            # Fit the encoder using training hotels ONLY.
            encoder=OneHotEncoder(handle_unknown='ignore')
            hid_tr=encoder.fit_transform(tr[[ID_HOTEL]].astype(str))
            hid_va=encoder.transform(validacao[[ID_HOTEL]].astype(str))
            Xtr_sp=sparse.hstack([sparse.csr_matrix(Xtr),hid_tr],format='csr')
            Xva_sp=sparse.hstack([sparse.csr_matrix(Xva),hid_va],format='csr')
            reg=Ridge(alpha=35.0,solver='lsqr')
            reg.fit(Xtr_sp,y_train)
            pred=reg.predict(Xva_sp)
        elif nome.startswith('ridge_'):
            reg=Ridge(alpha=12.0)
            reg.fit(Xtr,y_train);pred=reg.predict(Xva)
        elif nome.startswith('random_forest'):
            reg=RandomForestRegressor(n_estimators=140,min_samples_leaf=3,
                   n_jobs=MAX_THREADS,random_state=SEMENTE)
            reg.fit(Xtr,y_train);pred=reg.predict(Xva)
        elif nome.startswith('xgboost'):
            from xgboost import XGBRegressor
            reg=XGBRegressor(n_estimators=180,max_depth=4,learning_rate=.05,
                   subsample=.9,colsample_bytree=.9,objective='reg:squarederror',
                   tree_method='hist',n_jobs=MAX_THREADS,random_state=SEMENTE)
            reg.fit(Xtr,y_train);pred=reg.predict(Xva)
        elif nome=='lightgbm_arx':
            from lightgbm import LGBMRegressor
            reg=LGBMRegressor(n_estimators=200,learning_rate=.05,num_leaves=18,
                   max_depth=5,min_child_samples=20,verbosity=-1,
                   n_jobs=MAX_THREADS,random_state=SEMENTE)
            reg.fit(Xtr,y_train);pred=reg.predict(Xva)
        elif nome in {'tabm_static','tabm_arx','realmlp_arx'}:
            import pytabkit
            try:
                import torch
                device='cuda' if torch.cuda.is_available() else 'cpu'
            except ImportError:
                device='cpu'
            classe='TabM_D_Regressor' if nome.startswith('tabm') else 'RealMLP_TD_Regressor'
            modelo_cls=getattr(pytabkit,classe)
            # Conservative configuration for laptops; NOT the full version
            # of the training budgets used in the original publications.
            reg=modelo_cls(device=device,random_state=SEMENTE,n_threads=MAX_THREADS,
                            n_epochs=70,n_cv=1,n_refit=0,verbosity=0)
            reg.fit(Xtr,y_train);pred=reg.predict(Xva)
        else:
            raise ValueError('Unknown model: '+str(nome))
    pred=np.maximum(np.asarray(pred,dtype=float).reshape(-1),0)
    return pred,{'n_treino':len(tr),'limite_alvo_treino':limite,
                 'tempo_s':float(time.perf_counter()-inicio),'n_features':len(features),
                 'limite_lag_treino':limite_lag}

print('Training functions are ready.')

## 8. Main benchmark — fair year-by-year comparison (2021–2023)

The primary scenario is specified before evaluating the dynamic benchmark: `todos_p995`. The p99.5 option clips **only the training target** to reduce the extreme influence of anomalous values; all test sets retain the original observed values.

Models are evaluated on the **same 886/927/954 observations for the respective years**, according to historical availability; exact counts are verified in the output. The 2021–2023 years differ in difficulty, so we report results separately by year and aggregate RMSE relative to each year's Dummy baseline.


In [ ]:
linhas_cv=[]; previsoes_cv=[]
modelos_por_ano={}
for ano in ANOS_CV:
    tr,va=split_by_year(ano)
    for nome in MODELOS:
        try:
            pred,info=fit_and_predict(nome,tr,va,CENARIO_PRINCIPAL)
            m=calculate_metrics(va[COL_ALVO],pred)
            linhas_cv.append({'modelo':nome,'ano':ano,'cenario':CENARIO_PRINCIPAL,
                              'n_avaliacao':len(va),**m,**info})
            previsoes_cv.extend({'ht_id':hid,'ano':ano,'modelo':nome,
                'real':float(yy),'previsao':float(pp)}
                for hid,yy,pp in zip(va[ID_HOTEL],va[COL_ALVO],pred))
            print(ano,nome,'RMSE',round(m['RMSE'],2),'R2',round(m['R2'],3))
            modelos_por_ano[(nome,ano)]='EXECUTADO'
        except Exception as exc:
            record_failure('validacao',nome,ano,CENARIO_PRINCIPAL,exc)
            modelos_por_ano[(nome,ano)]='FALHOU'

cv=pd.DataFrame(linhas_cv)
pred_cv=pd.DataFrame(previsoes_cv)
if cv.empty: raise RuntimeError('No models ran. Check dependencies and the Excel file.')
# Normalise for year-level difficulty using a Dummy on the SAME fold sample.
dummy_rmse=cv.loc[cv.modelo=='media_treino',['ano','RMSE']].set_index('ano')['RMSE']
cv['RMSE_rel_dummy']=cv.apply(lambda z:z.RMSE/dummy_rmse.loc[z.ano],axis=1)
folds_ok=cv.groupby('modelo')['ano'].nunique()
modelos_completos=folds_ok[folds_ok==len(ANOS_CV)].index.tolist()
ranking=cv[cv.modelo.isin(modelos_completos)].groupby('modelo').agg(
    rmse_rel_medio=('RMSE_rel_dummy','mean'),rmse_medio=('RMSE','mean'),
    mae_medio=('MAE','mean'),r2_medio=('R2','mean'),n_folds=('ano','nunique')
).sort_values('rmse_rel_medio')
print('RANKING — models with all three folds completed only')
display(ranking.round(3))
print('Status incompletos:',sorted(set(MODELOS)-set(modelos_completos)))

## 9. Retrospective audit for 2024

**Scientific caution:** 2024 was already examined during development of earlier notebooks. Accordingly, this block reports retrospective results and must not be described in the paper as a perfectly blind evaluation. No decision in this notebook will be selected on the basis of 2024 results.

To predict 2024, we use only pairs with target years up to 2023, along with each hotel's information known in 2023.


In [ ]:
linhas_2024=[]; prev_2024=[]
if EXECUTAR_AUDITORIA_2024:
    treino24,avaliacao24=split_by_year(ANO_AUDITORIA)
    for nome in modelos_completos:
        try:
            pred,info=fit_and_predict(nome,treino24,avaliacao24,CENARIO_PRINCIPAL)
            mm=calculate_metrics(avaliacao24[COL_ALVO],pred)
            linhas_2024.append({'modelo':nome,'ano':ANO_AUDITORIA,'n_avaliacao':len(avaliacao24),
                                **mm,**info})
            prev_2024.extend({'ht_id':hid,'ano':ANO_AUDITORIA,'modelo':nome,
                 'real':float(y),'previsao':float(p)}
                 for hid,y,p in zip(avaliacao24[ID_HOTEL],avaliacao24[COL_ALVO],pred))
        except Exception as exc:
            record_failure('auditoria_2024',nome,ANO_AUDITORIA,CENARIO_PRINCIPAL,exc)

teste24=pd.DataFrame(linhas_2024)
pred24=pd.DataFrame(prev_2024)
if not teste24.empty:
    display(teste24.sort_values('RMSE')[['modelo','MAE','RMSE','R2','n_avaliacao']].round(3))
else:
    print('No 2024 results: audit disabled or all models failed.')

## 10. Figures — year-to-year stability and improvements from dynamic models

An isolated result for 2024 can be misleading. Here we compare **all three validation years**; the figure helps identify challenging periods and models that achieve consistent gains.


In [ ]:
modelos_fig=['persistencia','ridge_static','ridge_arx','ridge_panel_arx',
             'random_forest_static','random_forest_arx','random_forest_arx_lagclip',
             'xgboost_static','xgboost_arx','xgboost_arx_lagclip',
             'tabm_static','tabm_arx']
fig,ax=plt.subplots(figsize=(10.4,5.2))
for nome in modelos_fig:
    sub=cv.loc[cv.modelo==nome].sort_values('ano')
    if len(sub)==len(ANOS_CV):
        ax.plot(sub.ano,sub.RMSE,marker='o',linewidth=1.8,label=nome)
ax.set(xticks=ANOS_CV,title='Temporal validation — RMSE by year',xlabel='Forecast year',
       ylabel='RMSE (lower is better)')
ax.legend(ncol=2,fontsize=8);fig.tight_layout()
fig.savefig(PASTA_SAIDA/'figura_02_rmse_walkforward.png',dpi=190);plt.show()

pares_comparacao=[('ridge_static','ridge_arx'),
                  ('random_forest_static','random_forest_arx'),
                  ('xgboost_static','xgboost_arx'),
                  ('random_forest_static','random_forest_arx_lagclip'),
                  ('xgboost_static','xgboost_arx_lagclip'),
                  ('tabm_static','tabm_arx')]
comparacao=[]
for est,din in pares_comparacao:
    for ano in ANOS_CV:
        st=cv[(cv.modelo==est)&(cv.ano==ano)]
        dy=cv[(cv.modelo==din)&(cv.ano==ano)]
        if len(st)==1 and len(dy)==1:
            comparacao.append({'familia':din.replace('_arx',''),'ano':ano,
                'rmse_estatico':float(st.iloc[0].RMSE),'rmse_dinamico':float(dy.iloc[0].RMSE),
                'ganho_rmse':float(st.iloc[0].RMSE-dy.iloc[0].RMSE)})
comparacao_df=pd.DataFrame(comparacao)
if not comparacao_df.empty:
    display(comparacao_df.round(3))
    tab=comparacao_df.pivot(index='ano',columns='familia',values='ganho_rmse')
    ax=tab.plot(kind='bar',figsize=(9.5,4.5),rot=0)
    ax.axhline(0,color='black',lw=1)
    ax.set(title='Benefit from adding productivity at t−1 (positive = improvement)',
           ylabel='Static RMSE − dynamic RMSE')
    ax.figure.tight_layout();ax.figure.savefig(PASTA_SAIDA/'figura_03_ganho_lag.png',dpi=190)
    plt.show()

## 11. H1 — Paired comparison with hotel-level confidence intervals

To assess the effect of adding past productivity, the comparison must use **the same target observation for each hotel and year**. We calculate RMSE differences within each model family and, because hotels recur in multiple years, explore uncertainty using a bootstrap that resamples **entire hotels**, retaining all observations from each hotel.

The interval is exploratory: three years do not constitute a random sample of economic regimes. It should not be interpreted as a causal test or as automatically correcting the previous use of 2024.


In [ ]:
def clustered_gain_bootstrap(base_estatico,base_dinamico,n_boot=N_BOOTSTRAPS):
    a=pred_cv[pred_cv.modelo==base_estatico][['ht_id','ano','real','previsao']]
    b=pred_cv[pred_cv.modelo==base_dinamico][['ht_id','ano','previsao']]
    x=a.merge(b,on=['ht_id','ano'],validate='one_to_one',suffixes=('_est','_din'))
    if x.empty or len(x)!=len(a) or len(x)!=len(b):
        raise ValueError('Predictions do not correspond to the same hotel–year cohort.')
    x['se_est']=(x.real-x.previsao_est)**2
    x['se_din']=(x.real-x.previsao_din)**2
    por_hotel=x.groupby('ht_id').agg(se_est=('se_est','sum'),se_din=('se_din','sum'),n=('ano','size'))
    v=por_hotel[['se_est','se_din','n']].to_numpy(dtype=float)
    n_hotel=len(v);gerador=np.random.default_rng(SEMENTE)
    gains=[]
    for k in range(n_boot):
        idx=gerador.integers(0,n_hotel,size=n_hotel)
        somas=v[idx].sum(axis=0)
        gains.append(np.sqrt(somas[0]/somas[2])-np.sqrt(somas[1]/somas[2]))
    ganho=float(np.sqrt(x.se_est.mean())-np.sqrt(x.se_din.mean()))
    return {'RMSE_estatico':float(np.sqrt(x.se_est.mean())),
            'RMSE_dinamico':float(np.sqrt(x.se_din.mean())),
            'ganho_RMSE':ganho,'IC95_inf':float(np.quantile(gains,.025)),
            'IC95_sup':float(np.quantile(gains,.975)),
            'n_observacoes':len(x),'n_hoteis':n_hotel}

resultados_boot=[]
for est,din in pares_comparacao:
    if est in modelos_completos and din in modelos_completos:
        try:
            resultados_boot.append({'familia':din.replace('_arx',''),
                                    **clustered_gain_bootstrap(est,din)})
        except Exception as exc:
            record_failure('bootstrap',din,'2021-2023',CENARIO_PRINCIPAL,exc)
bootstrap_df=pd.DataFrame(resultados_boot)
if not bootstrap_df.empty:
    display(bootstrap_df.round(3))
    print('A 95% CI crossing zero indicates uncertain improvement under this bootstrap approximation.')

## 12. H2 — Sensitivity to the pandemic and the influence of extreme values

The **same dynamic-model family** is repeated under four prespecified policies. We do not change the validation set: every year contains exactly the same hotels. Comparing `todos_original` with `todos_p995` isolates target clipping; comparing `todos_p995` with `sem2020_p995` isolates the inclusion of 2020 in training.

This does not causally identify the effect of the pandemic. It is an assessment of model robustness to a period of regime change.


In [ ]:
sensibilidade=[]
if EXECUTAR_SENSIBILIDADE:
    MODELOS_SENSIBILIDADE=['ridge_arx','xgboost_arx']
    for nome in MODELOS_SENSIBILIDADE:
        for cenario in CENARIOS_SENSIBILIDADE:
            for ano in ANOS_CV:
                tr,va=split_by_year(ano)
                try:
                    p,info=fit_and_predict(nome,tr,va,cenario)
                    sensibilidade.append({'modelo':nome,'cenario':cenario,'ano':ano,
                                         'n':len(va),**calculate_metrics(va[COL_ALVO],p)})
                except Exception as exc:
                    record_failure('sensibilidade',nome,ano,cenario,exc)
sens_df=pd.DataFrame(sensibilidade)
if not sens_df.empty:
    print('RMSE by year and scenario:')
    display(sens_df.pivot_table(index=['modelo','cenario'],columns='ano',values='RMSE').round(2))
    for nome in sens_df.modelo.unique():
        q=sens_df[sens_df.modelo==nome].pivot(index='ano',columns='cenario',values='RMSE')
        ax=q.plot(marker='o',figsize=(8.5,3.9))
        ax.set(title='Temporal sensitivity — '+nome,xlabel='Forecast year',ylabel='RMSE')
        ax.figure.tight_layout()
        ax.figure.savefig(PASTA_SAIDA/('figura_04_sensibilidade_'+nome+'.png'),dpi=190)
        plt.show()

## 13. H3 — Are spatial features still useful after including temporal memory?

We compare `XGBoost ARX` with and without spatial features (beaches, churches, castles, and restaurants/cafés). We preserve **exactly the same hotels, years, lags, algorithm, and target treatment**.

Improved performance would indicate an **incremental predictive contribution** from spatial information. It would still not establish that opening more restaurants causes higher productivity; location and hotel quality may be associated with unobserved factors.


In [ ]:
resultados_espaco=[]
par_modelos=('xgboost_arx_sem_espaco','xgboost_arx')
if set(par_modelos).issubset(set(modelos_completos)):
    for ano in ANOS_CV:
        se=cv[(cv.modelo==par_modelos[0])&(cv.ano==ano)].iloc[0]
        co=cv[(cv.modelo==par_modelos[1])&(cv.ano==ano)].iloc[0]
        resultados_espaco.append({'ano':ano,'RMSE_sem_espaco':float(se.RMSE),
                 'RMSE_com_espaco':float(co.RMSE),'ganho_RMSE':float(se.RMSE-co.RMSE),
                 'R2_sem_espaco':float(se.R2),'R2_com_espaco':float(co.R2)})
espaco_df=pd.DataFrame(resultados_espaco)
if not espaco_df.empty:
    display(espaco_df.round(3))
    ax=espaco_df.plot(x='ano',y=['RMSE_sem_espaco','RMSE_com_espaco'],kind='bar',
                    figsize=(8.5,4),rot=0)
    ax.set(title='H3 — Incremental value of spatial context (XGBoost + lag)',
           xlabel='Forecast year',ylabel='RMSE')
    ax.figure.tight_layout();ax.figure.savefig(PASTA_SAIDA/'figura_05_ablacao_espacial.png',dpi=190)
    plt.show()

## 14. Statistical interpretability: ARX with hotel-clustered uncertainty

Beyond forecasting, an OLS-ARX regression can provide **exploratory associative coefficients**. To avoid using data after 2023, we fit this summary using training observations up to 2023 only. Standard errors are clustered by hotel.

**This is not a causal model:** omitted variables, endogeneity, and temporal disturbances remain possible. The p99.5-clipped-target version estimates a more robust association, not necessarily the relationship across the entire distribution tail.


In [ ]:
coef_df=pd.DataFrame()
try:
    import statsmodels.api as sm
    treino_inferencia=pares[pares[COL_ANO]<=2023].copy()
    tr_inf,y_inf,_=prepare_sample(treino_inferencia,CENARIO_PRINCIPAL)
    feats_inf=ATRIBUTOS_ARX
    X_inf,_=get_feature_matrices(tr_inf,tr_inf,feats_inf,standardize=True)
    xsm=sm.add_constant(X_inf,has_constant='add')
    fit_arx=sm.OLS(y_inf,xsm).fit(cov_type='cluster',
                       cov_kwds={'groups':tr_inf[ID_HOTEL].to_numpy()})
    coef_df=pd.DataFrame({'variavel':['intercepto']+feats_inf,
                         'coeficiente':fit_arx.params,'erro_padrao_agrupado':fit_arx.bse,
                         'p_valor_exploratorio':fit_arx.pvalues})
    display(coef_df.round(4))
    print('Coefficients per one predictor standard deviation (including the lag).')
    print('Caution: p-values are exploratory and do not establish causality.')
except Exception as exc:
    record_failure('interpretabilidade','OLS_ARX_2023',2023,CENARIO_PRINCIPAL,exc)

## 15. Fixed-effects panel — optional econometric demonstration

The panel is **short (T ≤ 7)**. We first estimate hotel fixed effects through a *within* transformation, using `statsmodels`, which is already among the requirements. If `linearmodels` is installed, we show a second confirmatory estimation using `PanelOLS`. We include only lagged productivity and a time trend, because nearly time-invariant spatial characteristics are absorbed by fixed effects.

**Methodological warning:** including a lagged dependent variable with a small T may produce Nickell bias. This estimation is a **diagnostic exercise**, will not enter the predictive ranking, and **is not a validated causal estimate**. Methods such as Arellano–Bond would require careful treatment of instrumentation, persistence, and the number of time periods.

Technical sources: https://bashtage.github.io/linearmodels/panel/introduction.html and https://bashtage.github.io/linearmodels/panel/examples/examples.html .


In [ ]:
painel_fe_df=pd.DataFrame()
if EXECUTAR_PAINEL_FE_DESCRITIVO:
    try:
        import statsmodels.api as sm
        sample_fe=pares[pares[COL_ANO]<=2023].copy()
        trfe,yfe,_=prepare_sample(sample_fe,CENARIO_PRINCIPAL)
        trfe['y_ajuste']=yfe
        ex_vars=['produtividade_lag1','ano_previsao_centrado']
        # Within transformation: each observation minus its hotel's mean,
        # computed EXCLUSIVELY from training observations up to 2023.
        for v in ex_vars+['y_ajuste']:
            trfe['within__'+v]=trfe[v]-trfe.groupby(ID_HOTEL)[v].transform('mean')
        Xt=trfe[['within__'+v for v in ex_vars]].to_numpy(dtype=float)
        yt=trfe['within__y_ajuste'].to_numpy(dtype=float)
        fe_fit=sm.OLS(yt,Xt).fit(cov_type='cluster',cov_kwds={'groups':trfe[ID_HOTEL].to_numpy()})
        painel_fe_df=pd.DataFrame({'metodo':'within_OLS','variavel':ex_vars,
            'coeficiente':fe_fit.params,'erro_padrao':fe_fit.bse})
        display(painel_fe_df.round(4))
        print('Descriptive within fixed-effects model fitted using statsmodels.')
        print('Short T + lagged dependent variable => possible Nickell bias; DO NOT interpret causally.')
    except Exception as exc:
        record_failure('painel_within','within_OLS',2023,CENARIO_PRINCIPAL,exc)
    if importlib.util.find_spec('linearmodels') is not None:
        try:
            from linearmodels.panel import PanelOLS
            te=trfe.set_index([ID_HOTEL,COL_ANO]).sort_index()
            ex=te[ex_vars].astype(float)
            model_fe=PanelOLS(te['y_ajuste'],ex,entity_effects=True,drop_absorbed=True)
            fit_fe=model_fe.fit(cov_type='clustered',cluster_entity=True)
            extra=pd.DataFrame({'metodo':'PanelOLS_linearmodels','variavel':fit_fe.params.index,
                  'coeficiente':fit_fe.params.values,'erro_padrao':fit_fe.std_errors.values})
            painel_fe_df=pd.concat([painel_fe_df,extra],ignore_index=True)
            display(extra.round(4))
        except Exception as exc:
            record_failure('painel_descritivo','PanelOLS',2023,CENARIO_PRINCIPAL,exc)
    else:
        print('linearmodels is not installed; the within-OLS above has already run.')
else:
    print('Fixed-effects diagnostic disabled.')

## 16. Examine error tails and atypical hotels (2024 audit only)

RMSE gives large errors quadratic weight. Productivity is skewed, so we evaluate performance by ranges of the **true observed value**; band thresholds are estimated from training data before 2024 only. These plots are **retrospective diagnostics**, not a basis for selecting new hyperparameters.


In [ ]:
faixas_df=pd.DataFrame(); outliers_2024=pd.DataFrame()
if not pred24.empty:
    q25,q75,q95=np.quantile(treino24[COL_ALVO],[.25,.75,.95])
    p2=pred24[pred24.modelo.isin(['xgboost_static','xgboost_arx','ridge_arx',
                                  'random_forest_arx'])].copy()
    p2['faixa']=pd.cut(p2.real,bins=[-np.inf,q25,q75,q95,np.inf],
                     labels=['baixo','intermedio','alto','extremo'],include_lowest=True)
    faixa=[]
    for (mod,fx),g in p2.groupby(['modelo','faixa'],observed=True):
        faixa.append({'modelo':mod,'faixa':str(fx),'n':len(g),
                      'MAE':mean_absolute_error(g.real,g.previsao),
                      'RMSE':np.sqrt(mean_squared_error(g.real,g.previsao))})
    faixas_df=pd.DataFrame(faixa)
    display(faixas_df.round(3))
    if 'xgboost_arx' in p2.modelo.values:
        g=p2[p2.modelo=='xgboost_arx'].copy()
        g['erro_absoluto']=abs(g.real-g.previsao)
        outliers_2024=g.sort_values('erro_absoluto',ascending=False).head(15).copy()
        display(outliers_2024[['ht_id','ano','real','previsao','erro_absoluto']].round(2))
    p1=p2[p2.modelo=='xgboost_arx']
    if len(p1)>0:
        fig,ax=plt.subplots(figsize=(6.6,5.6))
        ax.scatter(p1.real,p1.previsao,alpha=.4,s=12)
        mm=max(p1.real.max(),p1.previsao.max())
        ax.plot([0,mm],[0,mm],ls='--',lw=1,color='black')
        ax.set(title='XGBoost-ARX — observed vs predicted (2024, retrospective)',
               xlabel='Observed productivity',ylabel='Predicted productivity')
        fig.tight_layout();fig.savefig(PASTA_SAIDA/'figura_06_previsoes_2024.png',dpi=190);plt.show()

## 17. Export, execution log, and reproducibility

The notebook never invents results for libraries that are not installed. The CSV files contain the comparison cohort, validation years, errors, sensitivity analysis, and status of each model. The Excel file **is not modified** and is not included among the exported results.

For the version submitted to KBS, also document hardware, Python version, package versions, and the history of the spatial data collection (especially the date when restaurants/cafés were counted).


In [ ]:
tabelas={
 '01_auditoria_pares_por_ano':contagens.reset_index(),
 '02_resumo_produtividade_por_ano':por_ano,
 '03_validacao_temporal_modelos_2021_2023':cv,
 '04_ranking_cv_modelos_completos':ranking.reset_index(),
 '05_predicoes_cv_pares_hotel_ano':pred_cv,
 '06_comparacao_estatico_dinamico':comparacao_df,
 '07_bootstrap_agrupado_H1':bootstrap_df,
 '08_sensibilidade_pandemia_H2':sens_df,
 '09_ablacao_espacial_H3':espaco_df,
 '10_auditoria_2024':teste24,
 '11_previsoes_2024':pred24,
 '12_arx_ols_coeficientes_2023':coef_df,
 '13_fe_painel_descritivo':painel_fe_df,
 '14_erros_faixas_2024':faixas_df,
 '15_maiores_erros_2024':outliers_2024,
 '16_falhas_modelos':pd.DataFrame(FALHAS,columns=['etapa','modelo','ano','cenario','erro','tipo','traceback']),
}
for nome,df in tabelas.items():
    if isinstance(df,pd.DataFrame):
        df.to_csv(PASTA_SAIDA/(nome+'.csv'),index=False)
status=[]
for nome in MODELOS:
    completou=sorted(cv.loc[cv.modelo==nome,'ano'].unique().tolist())
    status.append({'modelo':nome,'folds_cv_concluidos':len(completou),'anos_cv':str(completou),
            '2024_executou':bool(not teste24.empty and nome in teste24.modelo.values),
            'estado':'CV_COMPLETA' if len(completou)==len(ANOS_CV) else 'INCOMPLETO'})
status_df=pd.DataFrame(status)
status_df.to_csv(PASTA_SAIDA/'17_status_execucao_modelos.csv',index=False)
manifesto={'estudo':'KBS 04 Dynamic Panel / ARX','excel_nome':FICHEIRO_DADOS.name,
 'folha':FOLHA,'n_pares':len(pares),'coorte':'consecutive years only, with valid targets in t and t−1',
 'avaliacao_2024':'retrospective; year already examined in earlier notebooks',
 'cenarios':CENARIOS_SENSIBILIDADE,'cenario_principal':CENARIO_PRINCIPAL,
 'anos_cv':ANOS_CV,'modelos_solicitados':MODELOS,
 'modelos_cv_completa':modelos_completos,'python':sys.version,
 'plataforma':platform.platform(),'semente':SEMENTE,
 'observacao':'Historical availability of data and spatial variables needs verification.'}
(PASTA_SAIDA/'18_manifesto.json').write_text(json.dumps(manifesto,ensure_ascii=False,indent=2),encoding='utf-8')
display(status_df)
print('Files saved in:',PASTA_SAIDA.resolve())
print('Observed training/test values are never silently replaced.')

## 18. Writing the results for *Knowledge-Based Systems*

**What we can claim after running the experiments:**

- **H1:** whether historical productivity helps for **the same hotels in the same years**, and the magnitude and uncertainty of that gain.
- **H2:** whether the treatment of 2020 and extreme values changes temporal robustness—distinguishing economic shocks from the statistical influence of outliers.
- **H3:** whether location features improve predictions **after** controlling for previous productivity.

**What we cannot yet claim:** a causal effect of proximity to services; the efficiency of a public intervention; definitive architectural superiority based on a single year; seasonality estimated from annual data; generalisation to hotels never observed during training without a dedicated evaluation; performance of advanced models that did not finish training.

**Limitations to state in the paper:**

1. **Short T and incomplete panel:** only seven years are available, and the dynamic comparison includes exclusively hotels with valid observations at *t* and *t−1*.
2. **Spatial variables:** confirm when points of interest were actually collected and updated. Distance to a castle is stable; the number of restaurants within 800 m can change from year to year.
3. **Operational dependence:** prediction for *t* assumes the financial results for *t−1* have already been published.
4. **2024 is not blind:** this year was inspected during development, so its metrics constitute retrospective checks. A later external validation would strengthen the contribution.
5. **Panel model:** penalising hotel-specific intercepts is a predictive strategy, and lagged fixed-effects models do not yield causal estimates without additional assumptions.

**Next editorial step:** prepare a consolidated table with validation results for 2021/2022/2023, the benefit of adding lagged productivity, spatial ablation, and diagnostics for extreme hotels. Scientific interpretation should rely on controlled comparisons, not solely on the largest R².
